# Project E.C.H.O. - Radar Target Artificial Intelligence

This notebook builds the ultra-lightweight Neural Network designed to operate on an **ESP8266 Microcontroller**.
It loads massive raw FMCW radar streams and perfectly compresses the logic into a native C++ format.

In [1]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.metrics import classification_report, confusion_matrix
print("E.C.H.O. Radar AI Environment Loaded.")

In [2]:
# Zenodo dataset details
DATASET_DOWNLOAD_URL = "https://zenodo.org/records/5845259/files/data_SAAB_SIRS_77GHz_FMCW.npy?download=1"
DATASET_FILENAME = "data_SAAB_SIRS_77GHz_FMCW.npy"

print(f"Downloading dataset...")
DATASET_PATH = tf.keras.utils.get_file(DATASET_FILENAME, DATASET_DOWNLOAD_URL)
data = np.load(DATASET_PATH, allow_pickle=True)

In [3]:
drones = ['D1', 'D2', 'D3', 'D4', 'D5', 'D6']
birds = ['seagull', 'pigeon', 'raven', 'black-headed gull', 'heron', 'seagull and black-headed gull']
humans = ['human_walk', 'human_run']

X_list = []
y_list = []
class_names = ['Bird', 'Drone', 'Human']

print("Extracting micro-Doppler signatures...")
for row in data:
    label_str = row[0]
    if label_str in birds: label = 0
    elif label_str in drones: label = 1
    elif label_str in humans: label = 2
    else: continue 

    complex_data = row[1]
    real_magnitude_matrix = np.abs(complex_data)
    for i in range(real_magnitude_matrix.shape[1]):
        X_list.append(real_magnitude_matrix[:, i])
        y_list.append(label)

X = np.array(X_list)
y = np.array(y_list)
X, y = shuffle(X, y, random_state=42)

In [4]:
global_max_val = np.max(X)
X = X / global_max_val
X = X.reshape((X.shape[0], X.shape[1], 1))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Dataset: {len(X_train)} Train | {len(X_test)} Test")
print(f"ESP8266 Normalization Factor: {global_max_val:.4f}")

In [5]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv1D(filters=32, kernel_size=16, strides=4, activation='relu', input_shape=(1280, 1)),
    tf.keras.layers.MaxPooling1D(pool_size=4),
    tf.keras.layers.Conv1D(filters=64, kernel_size=8, strides=4, activation='relu'),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)

print("Training Radar Intelligence Model...")
history = model.fit(X_train, y_train, epochs=50, batch_size=128, validation_data=(X_test, y_test), callbacks=[early_stop])

### 📊 Advanced Performance Analytics
Verifying Precision, Recall, and Accuracy for Bird/Drone/Human separation.

In [6]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("Radar Classification Report:")
print(classification_report(y_test, y_pred_classes, target_names=class_names))

cm = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
            xticklabels=class_names, 
            yticklabels=class_names)
plt.title('Radar Target Confusion Matrix')
plt.ylabel('Actual Class')
plt.xlabel('Predicted Class')
plt.show()

### 🛡️ ESP8266 Export
This logic converts the weight matrices into constant arrays for the NodeMCU C++ firmware.

In [7]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("echo_radar_model.h", "w") as f:
    f.write("const unsigned char echo_radar_model_tflite[] = {\n ")
    for i, byte in enumerate(tflite_model):
        f.write(f"{byte:#04x}")
        if i != len(tflite_model)-1: f.write(", ")
        if (i+1) % 12 == 0: f.write("\n ")
    f.write("\n};")

print(f"SUCCESS: Radar Model exported as echo_radar_model.h ({len(tflite_model)/1024:.1f} KB)")